In [1]:
!pip install open_clip_torch torch torchvision

In [2]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nUsing device: {device}")

print(f"\nGPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"\nGPU Name: {torch.cuda.get_device_name(0)}")


Using device: cuda

GPU Available: True

GPU Name: Tesla T4


In [3]:
import open_clip

print("\nAvailable CLIP models:")
print(open_clip.list_models())

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 


Available CLIP models:
['coca_base', 'coca_roberta-ViT-B-32', 'coca_ViT-B-32', 'coca_ViT-L-14', 'convnext_base', 'convnext_base_w', 'convnext_base_w_320', 'convnext_large', 'convnext_large_d', 'convnext_large_d_320', 'convnext_small', 'convnext_tiny', 'convnext_xlarge', 'convnext_xxlarge', 'convnext_xxlarge_320', 'EVA01-g-14', 'EVA01-g-14-plus', 'EVA02-B-16', 'EVA02-E-14', 'EVA02-E-14-plus', 'EVA02-L-14', 'EVA02-L-14-336', 'MobileCLIP2-B', 'MobileCLIP2-L-14', 'MobileCLIP2-S0', 'MobileCLIP2-S2', 'MobileCLIP2-S3', 'MobileCLIP2-S4', 'MobileCLIP-B', 'MobileCLIP-S1', 'MobileCLIP-S2', 'mt5-base-ViT-B-32', 'mt5-xl-ViT-H-14', 'nllb-clip-base', 'nllb-clip-base-siglip', 'nllb-clip-large', 'nllb-clip-large-siglip', 'PE-Core-B-16', 'PE-Core-bigG-14-448', 'PE-Core-L-14-336', 'PE-Core-S-16-384', 'PE-Core-T-16-384', 'RN50', 'RN50-quickgelu', 'RN50x4', 'RN50x4-quickgelu', 'RN50x16', 'RN50x16-quickgelu', 'RN50x64', 'RN50x64-quickgelu', 'RN101', 'RN101-quickgelu', 'roberta-ViT-B-32', 'swin_base_patch4_

In [4]:
import open_clip

model_name = "ViT-B-32"
print(f"\nLoading CLIP Model: {model_name}")

model, _, preprocess = open_clip.create_model_and_transforms(model_name, pretrained='openai')
tokenizer = open_clip.get_tokenizer(model_name)
model = model.to(device)

print(f"\nModel Loaded Successfully")


Loading CLIP Model: ViT-B-32


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(



Model Loaded Successfully


In [5]:
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from PIL import Image

class ImageTextDataset(Dataset):
    def __init__(self, csv_path, preprocess):
        df = pd.read_csv(csv_path)
        self.images = df['image_path'].tolist()
        self.texts = df['text'].astype(str).tolist()
        self.preprocess = preprocess

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = Image.open(self.images[idx]).convert("RGB")
        img = self.preprocess(img)
        txt = self.texts[idx]
        return img, txt

In [6]:
from torchvision.datasets import CocoCaptions
import os
import pandas as pd
from tqdm import tqdm
import kagglehub

path = kagglehub.dataset_download("awsaf49/coco-2017-dataset")
print(f"\nPath to Dataset Files: {path}")

coco2017_path = os.path.join(path, "coco2017")
coco_root = os.path.join(coco2017_path, "train2017")
ann_file = os.path.join(coco2017_path, "annotations/captions_train2017.json")
train_csv_path = "dataset_train.csv"

print()
coco = CocoCaptions(root=coco_root, annFile=ann_file)
print()

if not os.path.exists(train_csv_path):
    image_paths = []
    texts = []

    for idx in tqdm(range(len(coco)), desc="Processing COCO Captions"):
        img, captions = coco[idx]
        img_path = os.path.join(coco_root, coco.coco.imgs[coco.ids[idx]]['file_name'])
        for caption in captions:
            image_paths.append(img_path)
            texts.append(caption)

    df = pd.DataFrame({"image_path": image_paths, "text": texts})
    df.to_csv(train_csv_path, index=False)
    print(f"\nSaved CSV To: {train_csv_path}")
else:
    print(f"\nCSV File Already Exists: {train_csv_path}")
    df = pd.read_csv(train_csv_path)


Path to Dataset Files: /kaggle/input/coco-2017-dataset

loading annotations into memory...
Done (t=1.43s)
creating index...
index created!



Processing COCO Captions: 100%|██████████| 118287/118287 [25:21<00:00, 77.73it/s]



Saved CSV To: dataset_train.csv


In [7]:
df = df.groupby('image_path').apply(lambda x: x.sample(1)).reset_index(drop=True)

/tmp/ipykernel_121/766484715.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('image_path').apply(lambda x: x.sample(1)).reset_index(drop=True)


In [8]:
df['image_id'] = df['image_path'].apply(lambda x: int(x.split('/')[-1].split('.')[0]))


In [9]:
df_subset = df.sample(n=5000, random_state=42).reset_index(drop=True)


###ICT-Q


In [10]:
# df_subset_ictq = df_subset.drop(columns=["neg_img_idx"])
df_subset_ictq = df_subset

In [11]:
!pip install torch torchvision transformers scikit-learn faiss-cpu pandas pillow


In [12]:
import os
import random
from tqdm import tqdm
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import numpy as np
import hdbscan
import pandas as pd

# ----------------------------
# USER CONFIG
# ----------------------------
# Assumes df_subset is already in your namespace. Otherwise load it:
# df_subset = pd.read_csv("your_df.csv")  # if needed

df = df_subset_ictq.reset_index(drop=True)  # <- your dataframe
assert "image_path" in df.columns and "text" in df.columns

MODEL_NAME = "openai/clip-vit-base-patch32"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

EPOCHS = 7
BATCH_SIZE = 16
LR = 2e-5
WEIGHT_DECAY = 0.01
CLUSTER_EVERY_N_EPOCHS = 1   # ICT-Q: refresh clusters every n epochs
MIN_CLUSTER_SIZE = 5         # HDBSCAN parameter: minimum queries per cluster
MIN_SAMPLES = 3              # HDBSCAN parameter: affects noise detection
N_HARD_NEG_PER_QUERY = 7     # number of hard negatives to sample per query
SEED = 67
SAVE_DIR = "./clip_ictq_model"
os.makedirs(SAVE_DIR, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ----------------------------
# MODEL + PROCESSOR
# ----------------------------
processor = CLIPProcessor.from_pretrained(MODEL_NAME)
model = CLIPModel.from_pretrained(MODEL_NAME).to(DEVICE)
model.train()  # training mode

# ----------------------------
# SIMPLE DATASET / DATALOADER
# ----------------------------
class CocoSubsetDataset(Dataset):
    def __init__(self, df):
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return {
            "image_path": row["image_path"],
            "text": str(row["text"]),
            "idx": int(idx),
        }

dataset = CocoSubsetDataset(df)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)

# ----------------------------
# HELPERS: compute all text embeddings (uses current model)
# ----------------------------
@torch.no_grad()
def compute_text_embeddings(model, processor, texts, batch_size=64, device=DEVICE):
    model.eval()
    embs = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        inputs = processor(text=batch_texts, return_tensors="pt", padding=True, truncation=True).to(device)
        text_feats = model.get_text_features(**inputs)  # (B, D)
        # normalize (useful for cosine similarity)
        text_feats = text_feats / text_feats.norm(dim=-1, keepdim=True)
        embs.append(text_feats.cpu().float())
    embs = torch.cat(embs, dim=0)
    model.train()
    return embs.numpy()  # return numpy for sklearn

def build_clusters(embeddings, min_cluster_size=MIN_CLUSTER_SIZE, min_samples=MIN_SAMPLES):
    # Use HDBSCAN to cluster query embeddings
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        metric='euclidean',
        cluster_selection_method='eom'
    )
    cluster_ids = clusterer.fit_predict(embeddings)
    return cluster_ids



# ----------------------------
# Prepare initial clusters (ICT-Q initialization)
# ----------------------------
all_texts = df["text"].astype(str).tolist()
print("Computing initial text embeddings (this uses the CURRENT model weights)...")
text_embs = compute_text_embeddings(model, processor, all_texts, batch_size=64)
cluster_ids = build_clusters(text_embs, min_cluster_size=MIN_CLUSTER_SIZE, min_samples=MIN_SAMPLES)
# Build cluster -> list of indices mapping for sampling
from collections import defaultdict
def build_cluster_map(cluster_ids):
    cluster_map = defaultdict(list)
    for i, c in enumerate(cluster_ids):
        cluster_map[int(c)].append(i)
    return cluster_map

cluster_map = build_cluster_map(cluster_ids)

# ----------------------------
# TRAINING: contrastive loss with ICT-Q hard negatives
# ----------------------------
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
criterion = nn.CrossEntropyLoss()
temperature = 1.0  # you can scale logits by 1/temp

global_step = 0
for epoch in range(EPOCHS):
    epoch_loss = 0.0
    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for batch in pbar:
        # batch contains list of image paths, texts, and indices
        image_paths = batch["image_path"]
        texts = batch["text"]
        idxs = batch["idx"].numpy()  # indices into df (0..N-1)
        B = len(texts)

        # Build the text pool: positives (batch texts first), then per-query hard negatives
        # collect hard negatives indices for each query
        hard_neg_indices = []
        for i, global_idx in enumerate(idxs):
            c = int(cluster_ids[global_idx])
            candidates = cluster_map[c]
            # exclude the positive index itself
            candidates = [x for x in candidates if x != global_idx]
            if len(candidates) >= N_HARD_NEG_PER_QUERY:
                sampled = random.sample(candidates, N_HARD_NEG_PER_QUERY)
            else:
                # not enough in-cluster candidates: fallback sample from entire dataset excluding positives
                pool = list(range(len(df)))
                pool = [x for x in pool if x != global_idx]
                sampled = random.sample(pool, N_HARD_NEG_PER_QUERY)
            hard_neg_indices.extend(sampled)

        # Build text_pool: batch positives first, then hard negatives grouped by query
        text_pool_texts = []
        # positives
        text_pool_texts.extend(list(texts))  # size B
        # hard negatives (ordered: for query0 -> H negs, query1 -> H negs,...)
        hard_texts = [df.loc[i, "text"] for i in hard_neg_indices]
        text_pool_texts.extend(hard_texts)  # size B*H

        # Build images list for the batch (we only have B images)
        # Process inputs using processor (returns tensors)
        # For images, open and convert
        imgs = [Image.open(p).convert("RGB") for p in image_paths]

        inputs = processor(text=text_pool_texts, images=imgs, return_tensors="pt", padding=True, truncation=True).to(DEVICE)

        # Get image and text features
        outputs = model(**inputs)
        # CLIP returns text features corresponding to the provided text list and image features for the provided images
        text_features = outputs.text_embeds  # shape: (B*(1+H), D)  NOTE: huggingface may return .text_embeds or .text_model_output - this uses .text_embeds attr
        image_features = outputs.image_embeds  # shape: (B, D)
        # If your version of transformers returns .text_model_output or different attr, replace accordingly.
        # normalize
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)

        # Split text_features into positives (first B rows) and rest are hard negatives
        pool_size = text_features.size(0)
        # compute logits: (B, pool_size)
        logits = image_features @ text_features.t()  # dot product
        logits = logits / temperature

        # labels: for image i, the positive text is at index i (first B entries)
        labels = torch.arange(B, device=DEVICE)

        loss_img_to_text = criterion(logits, labels)
        # also compute text->image loss (symmetric)
        logits_t2i = logits.t()  # (pool_size, B)
        # For text->image we want only the rows corresponding to the positives (first B)
        logits_pos_texts = logits_t2i[:B, :]  # (B, B)
        labels_t2i = torch.arange(B, device=DEVICE)
        loss_text_to_img = criterion(logits_pos_texts, labels_t2i)

        loss = (loss_img_to_text + loss_text_to_img) / 2.0

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        global_step += 1
        pbar.set_postfix({"loss": f"{(epoch_loss/global_step):.4f}"})

    avg_loss = epoch_loss / (global_step if global_step>0 else 1)
    print(f"Epoch {epoch+1} finished. Avg loss (running): {avg_loss:.4f}")

    # ----------------------------
    # ICT-Q: Refresh clusters/embeddings every CLUSTER_EVERY_N_EPOCHS
    # ----------------------------
    if (epoch + 1) % CLUSTER_EVERY_N_EPOCHS == 0:
        print(f"Recomputing text embeddings & reclustering (epoch {epoch+1})...")
        all_texts = df["text"].astype(str).tolist()
        text_embs = compute_text_embeddings(model, processor, all_texts, batch_size=64)
        cluster_ids = build_clusters(text_embs, min_cluster_size=MIN_CLUSTER_SIZE, min_samples=MIN_SAMPLES)
        cluster_map = build_cluster_map(cluster_ids)
        print("Re-clustering done.")

    # Save a checkpoint
    ckpt_path = os.path.join(SAVE_DIR, f"checkpoint_epoch_{epoch+1}.pt")
    torch.save(model.state_dict(), ckpt_path)
    print(f"Saved checkpoint to {ckpt_path}")

# final save
model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
print("Training complete. Model saved.")

2025-11-13 10:27:04.163597: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763029624.619814     121 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763029624.730194     121 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

... (21 outputs left out)

Epoch 6/7: 100%|██████████| 313/313 [03:20<00:00,  1.56it/s, loss=0.4960]


Epoch 6 finished. Avg loss (running): 0.4960
Recomputing text embeddings & reclustering (epoch 6)...
Re-clustering done.
Saved checkpoint to ./clip_ictq_model/checkpoint_epoch_6.pt


Epoch 7/7: 100%|██████████| 313/313 [03:22<00:00,  1.54it/s, loss=0.4267]


Epoch 7 finished. Avg loss (running): 0.4267
Recomputing text embeddings & reclustering (epoch 7)...
Re-clustering done.
Saved checkpoint to ./clip_ictq_model/checkpoint_epoch_7.pt
Training complete. Model saved.


In [13]:
from torchvision.datasets import CocoCaptions
import os
import pandas as pd
from tqdm import tqdm
import kagglehub

path = kagglehub.dataset_download("awsaf49/coco-2017-dataset")
print(f"\nPath to Dataset Files: {path}")

coco2017_path = os.path.join(path, "coco2017")
coco_root = os.path.join(coco2017_path, "train2017")
ann_file = os.path.join(coco2017_path, "annotations/captions_train2017.json")
train_csv_path = "dataset_train.csv"

print()
coco = CocoCaptions(root=coco_root, annFile=ann_file)
print()

if not os.path.exists(train_csv_path):
    image_paths = []
    texts = []

    for idx in tqdm(range(len(coco)), desc="Processing COCO Captions"):
        img, captions = coco[idx]
        img_path = os.path.join(coco_root, coco.coco.imgs[coco.ids[idx]]['file_name'])
        for caption in captions:
            image_paths.append(img_path)
            texts.append(caption)

    df_og = pd.DataFrame({"image_path": image_paths, "text": texts})
    df_og.to_csv(train_csv_path, index=False)
    print(f"\nSaved CSV To: {train_csv_path}")
else:
    print(f"\nCSV File Already Exists: {train_csv_path}")
    df_og = pd.read_csv(train_csv_path)


Path to Dataset Files: /kaggle/input/coco-2017-dataset

loading annotations into memory...
Done (t=1.46s)
creating index...
index created!


CSV File Already Exists: dataset_train.csv


In [14]:
df_og = df_og.groupby('image_path').apply(lambda x: x.sample(1)).reset_index(drop=True)

/tmp/ipykernel_121/763703059.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_og = df_og.groupby('image_path').apply(lambda x: x.sample(1)).reset_index(drop=True)


In [15]:
# Step 1: Remove the train subset from the full dataframe
df_remaining = df_og.drop(df_subset.index)

# Step 2: Sample exactly 1,000 rows for test set
df_test = df_remaining.sample(n=1000, random_state=42)

# Optional: Reset indices
df_test = df_test.reset_index(drop=True)
df_remaining = df_remaining.reset_index(drop=True)

print("Train size:", len(df_subset))
print("Test size:", len(df_test))
print("Remaining (unused):", len(df_remaining))


Train size: 5000
Test size: 1000
Remaining (unused): 113287


In [16]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
import numpy as np
from tqdm import tqdm

# ----------------------------
# Config
# ----------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 32

# Load your trained model
model_path = "./clip_ictq_model"
model = CLIPModel.from_pretrained(model_path).to(DEVICE)
processor = CLIPProcessor.from_pretrained(model_path)
model.eval()

# ----------------------------
# Dataset for testing
# ----------------------------
class TestDataset(Dataset):
    def __init__(self, df):
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return {
            "image_path": row["image_path"],
            "text": str(row["text"]),
            "idx": int(idx)
        }

test_dataset = TestDataset(df_test)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# ----------------------------
# Compute embeddings
# ----------------------------
all_image_embeds = []
all_text_embeds = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Computing embeddings"):
        # Images
        images = [Image.open(p).convert("RGB") for p in batch["image_path"]]
        texts = batch["text"]

        inputs = processor(
            text=texts,
            images=images,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(DEVICE)

        outputs = model(**inputs)

        # Normalize embeddings
        img_embeds = outputs.image_embeds / outputs.image_embeds.norm(dim=-1, keepdim=True)
        txt_embeds = outputs.text_embeds / outputs.text_embeds.norm(dim=-1, keepdim=True)

        all_image_embeds.append(img_embeds.cpu())
        all_text_embeds.append(txt_embeds.cpu())

# Concatenate embeddings
all_image_embeds = torch.cat(all_image_embeds, dim=0)  # shape: (num_images, dim)
all_text_embeds = torch.cat(all_text_embeds, dim=0)    # shape: (num_texts, dim)

# ----------------------------
# Compute similarity: Text -> Image
# ----------------------------
similarity = all_text_embeds @ all_image_embeds.T  # (num_texts, num_images)
labels = torch.arange(len(df_test))  # ground-truth indices

# ----------------------------
# Compute Text->Image retrieval metrics
# ----------------------------
def compute_retrieval_metrics(similarity, labels):
    num_queries = similarity.size(0)
    ranks = []
    rr = []

    for i in range(num_queries):
        sim_row = similarity[i]  # similarity scores for text i
        sorted_indices = torch.argsort(sim_row, descending=True)
        rank = (sorted_indices == labels[i]).nonzero(as_tuple=True)[0].item()
        ranks.append(rank + 1)
        rr.append(1.0 / (rank + 1))

    ranks = np.array(ranks)
    rr = np.array(rr)

    r1 = np.mean(ranks <= 1)
    r5 = np.mean(ranks <= 5)
    r10 = np.mean(ranks <= 10)
    medr = np.median(ranks)
    mrr = np.mean(rr)

    return {"R@1": r1, "R@5": r5, "R@10": r10, "MedR": medr, "MRR": mrr}

metrics_t2i = compute_retrieval_metrics(similarity, labels)

print("Text -> Image Retrieval Metrics:")
for k, v in metrics_t2i.items():
    if k.startswith("R@"):
        print(f"{k}: {v*100:.2f}%")
    else:
        print(f"{k}: {v:.2f}")


Computing embeddings: 100%|██████████| 32/32 [00:17<00:00,  1.83it/s]


Text -> Image Retrieval Metrics:
R@1: 8.90%
R@5: 27.20%
R@10: 39.50%
MedR: 16.00
MRR: 0.19
